# Launch & finish DPS runs of `run_extract_atl08_covars` (v2)

Extracts covariate pixel values (**Topo**, **HLS**, **Sentinel-1**) to the filtered 30 m ATL08 **v007** parquets for each tile and year. The outputs (`*_covars_NNNNNN.parquet`) are the training data for `run_boreal_biomass_map`. The notebook finds which tiles are still **missing** or **failed**, resubmits only those, and rebuilds the tile index (tindex) of the outputs.

**Run top to bottom. The only cell you edit is `1. Run configuration`.** Nothing is submitted unless `DRY_RUN = False`.

| Step | What it does | Touches DPS? |
|---|---|---|
| 2. Band check | For a sample tile per year, lists each covariate's bands and the column names they will get; flags missing bands and S1 units | no |
| 3. Test one tile | Optional: runs the extraction here for one tile and shows the columns and how many rows got values | no |
| 4. Status | ATL08 inputs vs extracted outputs per year: tiles **have** vs **missing** | no |
| 5. Fails | Latest job per tile from saved submission logs → which tiles **failed** (and still have no output) | reads job status |
| 6. Plan | Per year: tiles to submit (`DO_MISSING` / `DO_FAILS`), minus tiles that lack a covariate. Saved as CSV. | no |
| 7. Submit | Submits the plan (only if `DRY_RUN = False`) | **yes** |
| 8. Assess | Status counts for the submission just made | reads job status |
| 9. Rebuild tindex | After jobs finish: backs up, rebuilds the `ATL08_filt_extract` tindex from **all** run dates, verifies the count did not shrink | no |

**Sentinel-1 band names.** Column names come from each raster's band descriptions. The 2019/2020 S1 stacks used `vv_median_frozen … n_shoulder`, and those are the names in the model's `pred_vars`. Stacks with other names (e.g. `VV_frozen`, `frozen_VH`, `count_summer`) are mapped to those names automatically by `extract_atl08_covars.resolve_bandnames`. Anything it can't resolve, or a band with no description, is set in `S1_BAND_RENAME`. Section 2 shows the mapping before anything is submitted.

**Before the first run of this version** the algorithm must be registered (Appendix): `run_extract_atl08_covars` version `extract_atl08_covars_v2` adds the optional `band_rename` input and the S1 name handling. DPS builds from the git ref named by the version, so push the `extract_atl08_covars_v2` tag first.

Paul Montesano, PhD — reorganized Oct 2026 from `DPS_extract_covars.ipynb`, same layout as `DPS_run_boreal_v2.ipynb`

In [ ]:
import os, sys, glob, json, shutil, datetime, subprocess, importlib
from functools import lru_cache
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio as rio
from rasterio.windows import Window
import s3fs

from maap.maap import MAAP
maap = MAAP(maap_host='api.maap-project.org')

# One clone for imports AND for the build_tindex script
ICESAT2_BOREAL_REPO_PATH = '/projects/my-private-bucket/code/icesat2_boreal'
sys.path.insert(0, ICESAT2_BOREAL_REPO_PATH + '/lib')
import ExtractUtils
import CovariateUtils
import mosaiclib
import extract_atl08_covars
importlib.reload(extract_atl08_covars)
from mosaiclib import HLS_TINDEX_FN_DICT, TOPO_TINDEX_FN_DICT, SAR_TINDEX_FN_DICT, boreal_tile_index_path
print('extract_atl08_covars from', extract_atl08_covars.__file__)
s3 = s3fs.S3FileSystem(anon=True)

## 1. Run configuration — the only cell to edit
Fill-in runs **must** use the same `MAAP_VERSION`, covariates and `S1_BAND_RENAME` as the original run for that year, so all tiles of a year have the same columns.

In [ ]:
# --- What product -----------------------------------------------------------
ALGO_ID      = 'run_extract_atl08_covars'
MAAP_VERSION = 'extract_atl08_covars_v2'     # registered algorithm version (= git tag)
USER         = 'montesano'
WORKER_TYPE  = 'maap-dps-gedi_boreal_worker-8gb'
ATL08_VERSION = 'v007'
YEAR_LIST    = [str(y) for y in range(2019, 2026 + 1)]

# --- Inputs -------------------------------------------------------------------
ATL08_IN_ROOT = f'maap-ops-workspace/montesano/data/process_atl08_boreal/030m/{ATL08_VERSION}'   # {ATL08_IN_ROOT}/{year}/*_NNNNNN.parquet
TOPO_TINDEX   = TOPO_TINDEX_FN_DICT['c2020updated_v2']
HLS_TINDEX_BY_YEAR = {y: HLS_TINDEX_FN_DICT.get(y) for y in YEAR_LIST}   # check comps are good (Review_maps.ipynb, REDO lists in DPS_mscomp_stack.ipynb)

# Sentinel-1 tindex per year. 2019/2020 come from mosaiclib; add the new S1 stacks here
# (their bands can have different names; section 2 shows how they will be named).
INCLUDE_SAR  = True
SAR_TINDEX_BY_YEAR = {y: SAR_TINDEX_FN_DICT.get(y) for y in YEAR_LIST}
# SAR_TINDEX_BY_YEAR.update({
#     '2021': 's3://maap-ops-workspace/shared/montesano/DPS_tile_lists/.../S1_tindex_master.csv',
# })
SAR_REQUIRED = True    # True: tiles/years without S1 are not submitted. False: they run without S1 columns
                       # (then that year's training data lacks the S1 pred_vars for those tiles)

# Explicit band renames for names the automatic S1 mapping can't resolve: {band description or 'b<N>': new name}
S1_BAND_RENAME = {}    # e.g. {'b7': 'n_frozen', 'VVwinter': 'vv_median_frozen'}

# Column names the model expects (used only for the checks in section 2)
EXPECTED_BANDS = {
    'topo': ['elevation', 'slope', 'tsri', 'tpi'],
    'hls':  ['Red', 'Green', 'NIR', 'SWIR', 'SWIR2', 'NDVI', 'SAVI', 'MSAVI', 'NDMI', 'EVI', 'NBR', 'NBR2', 'TCB', 'TCG', 'TCW'],
    'sar':  extract_atl08_covars.S1_CANONICAL,
}
SAR_REFERENCE_TINDEX = SAR_TINDEX_FN_DICT['2020']   # S1 stacks the models were trained with: units check in section 2

# --- Which tiles -------------------------------------------------------------
DO_MISSING = True        # input parquet exists but no output yet
DO_FAILS   = False       # latest job failed and still no output
CONFIRM_FULL_RUN = False # must be True to submit every input tile (both flags False)
TILE_SET = None          # None = every tile with an ATL08 input; or a list of tile_num
REFRESH_FAILS = True     # re-query job status (slow for many jobs); False = reuse the cached fails JSON

# --- Safety -----------------------------------------------------------------
DRY_RUN  = True          # True: build and save the plan only. False: submit in step 7.
MAX_JOBS = 40000         # refuse to submit more than this in one go

# --- Paths --------------------------------------------------------------------
DPS_SUBMISSION_RESULTS_DIR = '/projects/my-public-bucket/dps_submission_results'
TINDEX_ROOT     = f'/projects/my-public-bucket/DPS_tile_lists/ATL08/{MAAP_VERSION}'   # build_tindex output root
DPS_OUTPUT_ROOT = f'/projects/my-private-bucket/dps_output/{ALGO_ID}/{MAAP_VERSION}'
PLAN_DIR        = f'{DPS_SUBMISSION_RESULTS_DIR}/plans'
TEST_OUTDIR     = '/projects/my-public-bucket/local_output/extract_atl08_covars_v2_test'
for d in (DPS_SUBMISSION_RESULTS_DIR, PLAN_DIR):
    os.makedirs(d, exist_ok=True)

In [ ]:
# Derived names (do not edit)
IDENTIFIERS = {y: f'atl08_{ATL08_VERSION}/{y}' for y in YEAR_LIST}
BAND_RENAME_STR = ','.join(f'{k}:{v}' for k, v in S1_BAND_RENAME.items())   # passed to DPS; no spaces allowed
assert ' ' not in BAND_RENAME_STR, 'S1_BAND_RENAME names cannot contain spaces'
MODE = '+'.join([m for m, on in (('MISSING', DO_MISSING), ('FAILS', DO_FAILS)) if on]) or 'ALL'
print(f'{ALGO_ID} {MAAP_VERSION} | ATL08 {ATL08_VERSION} | years {YEAR_LIST[0]}–{YEAR_LIST[-1]} | mode {MODE} | DRY_RUN={DRY_RUN}')
for y in YEAR_LIST:
    covs = ['topo'] + (['hls'] if HLS_TINDEX_BY_YEAR[y] else []) + (['sar'] if INCLUDE_SAR and SAR_TINDEX_BY_YEAR.get(y) else [])
    print(f'   {IDENTIFIERS[y]:22s} covariates: {" ".join(covs):14s}' + ('' if HLS_TINDEX_BY_YEAR[y] else '  !! no HLS tindex for this year'))

### Helper functions (no edits needed)

In [ ]:
def tile_of(fn):
    return int(os.path.basename(fn).split('_')[-1].split('.')[0])

@lru_cache(maxsize=None)
def list_inputs(year):
    """ATL08 input parquets for a year: {tile_num: s3 path}."""
    fns = s3.glob(f'{ATL08_IN_ROOT}/{year}/**/*.parquet')
    return {tile_of(f): 's3://' + f for f in fns}

def list_outputs(identifier):
    """Extracted outputs already written for an identifier: {tile_num: s3 path} (newest wins)."""
    fns = sorted(s3.glob(f'maap-ops-workspace/{USER}/dps_output/{ALGO_ID}/{MAAP_VERSION}/{identifier}/**/*_covars_*.parquet'))
    return {tile_of(f): 's3://' + f for f in fns}

def covar_tindexes(year):
    """{covariate: tindex csv} used for a year."""
    d = {'topo': TOPO_TINDEX, 'hls': HLS_TINDEX_BY_YEAR.get(year)}
    if INCLUDE_SAR:
        d['sar'] = SAR_TINDEX_BY_YEAR.get(year)
    return d

@lru_cache(maxsize=None)
def tindex_tiles(tindex_fn):
    return set(pd.read_csv(tindex_fn).tile_num.astype(int)) if tindex_fn else set()

def tindex_list_for_tile(tile_num, year):
    """Covariate tindexes to pass for this tile, or raise KeyError naming the missing covariate."""
    out = []
    for cov, fn in covar_tindexes(year).items():
        if fn is None or tile_num not in tindex_tiles(fn):
            if cov == 'sar' and not SAR_REQUIRED:
                continue
            raise KeyError(f'{cov}' + ('' if fn else ' (no tindex for year)'))
        out.append(fn)
    return out

def get_in_param_dict(tile_num, year):
    return {'s3_atl08_gdf_fn': list_inputs(year)[tile_num],
            'tindex_fn_list': ' '.join(tindex_list_for_tile(tile_num, year)),
            'band_rename': BAND_RENAME_STR}

In [ ]:
def stack_for_tile(tindex_fn, tile_num):
    try:
        return CovariateUtils.get_stack_fn(tindex_fn, tile_num, user=None, col_name='local_path')
    except IndexError:
        return None

def band_values_sample(fn, n=512):
    """Median of valid values per band in a central n x n window (enough to tell dB from power)."""
    with rio.env.Env(AWS_NO_SIGN_REQUEST='YES'), rio.open(fn) as src:
        w = Window(max(0, src.width // 2 - n // 2), max(0, src.height // 2 - n // 2), min(n, src.width), min(n, src.height))
        a = src.read(window=w, masked=True).astype('float64')
        return [float(np.ma.median(b)) if b.count() else np.nan for b in a]

def s1_units(v):
    if np.isnan(v):
        return 'no data'
    return 'dB?' if v < 0 else ('power' if v < 1 else '?')

def band_check(year, tile_num):
    """One row per band: description -> column name, and whether expected bands are present."""
    rows, missing = [], {}
    for cov, fn in covar_tindexes(year).items():
        if fn is None:
            missing[cov] = 'no tindex for year'; continue
        stack = stack_for_tile(fn, tile_num)
        if stack is None:
            missing[cov] = f'tile {tile_num} not in tindex'; continue
        names, table = extract_atl08_covars.resolve_bandnames(stack, S1_BAND_RENAME)
        table.insert(0, 'covar', cov)
        if cov == 'sar':
            table['median_value'] = band_values_sample(stack)
            table['units'] = [s1_units(v) if c.startswith('v') else '' for v, c in zip(table.median_value, table.column)]
        rows.append(table)
        lacking = [b for b in EXPECTED_BANDS.get(cov, []) if b not in names]
        if lacking:
            missing[cov] = f'expected bands not found: {lacking}'
    return (pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()), missing

In [ ]:
def submission_logs(identifier):
    """All saved submission CSVs for this identifier, oldest first (by the timestamp in the file name)."""
    s = identifier.replace('/', '_')
    fns = glob.glob(f'{DPS_SUBMISSION_RESULTS_DIR}/DPS_{ALGO_ID}_{MAAP_VERSION}_{s}_submission_results_*.csv')
    return sorted(fns, key=lambda f: int(os.path.splitext(f)[0].split('_')[-1]))

def latest_jobs(identifier):
    """One row per tile: the most recent submission for this identifier."""
    parts = []
    for order, fn in enumerate(submission_logs(identifier)):
        d = pd.read_csv(fn)
        d['log_file'], d['log_order'] = os.path.basename(fn), order
        parts.append(d)
    if not parts:
        return pd.DataFrame(columns=['tile_num', 'job_id', 'submit_status', 'log_file'])
    d = pd.concat(parts, ignore_index=True)
    d['tile_num'] = d.tile_num.astype(int)
    return d.sort_values(['log_order', 'dps_num']).drop_duplicates('tile_num', keep='last').reset_index(drop=True)

def job_status_table(jobs):
    """Adds a 'status' column via MAAP. Tiles that never got a job id are marked 'SubmitFailed'."""
    ok = jobs.job_id.notna() & (jobs.job_id.astype(str) != '')
    out = jobs.copy()
    out['status'] = 'SubmitFailed'
    if ok.any():
        st = ExtractUtils.BUILD_TABLE_JOBSTATUS(jobs[ok][['tile_num', 'job_id']])
        out.loc[ok, 'status'] = out.loc[ok, 'job_id'].map(dict(zip(st.job_id, st.status))).fillna('Unknown')
    return out

def build_plan(year, identifier, tile_set, have, failed):
    """Tiles to submit for one identifier, with the reason each tile is (or is not) included."""
    inputs = list_inputs(year)
    missing = sorted(set(tile_set) - have)
    want = set()
    if DO_MISSING:
        want |= set(missing)
    if DO_FAILS:
        want |= set(failed)
    if not (DO_MISSING or DO_FAILS):
        want = set(tile_set)
    rows = []
    for t in sorted(want):
        n_cov = None
        if t not in inputs:
            runnable, why = False, 'no ATL08 input parquet'
        else:
            try:
                n_cov = len(tindex_list_for_tile(t, year))
                runnable, why = True, ''
            except KeyError as e:
                runnable, why = False, f'no {e.args[0]} for tile'
        rows.append({'identifier': identifier, 'year': year, 'tile_num': t, 'is_missing': t in missing,
                     'is_failed': t in failed, 'has_output': t in have, 'n_covars': n_cov,
                     'runnable': runnable, 'reason_not_runnable': why})
    return pd.DataFrame(rows, columns=['identifier', 'year', 'tile_num', 'is_missing', 'is_failed', 'has_output',
                                       'n_covars', 'runnable', 'reason_not_runnable'])

def submit_identifier(identifier, year, tiles):
    rows = []
    for i, t in enumerate(tiles, 1):
        try:
            r = maap.submitJob(identifier=identifier, algo_id=ALGO_ID, version=MAAP_VERSION, username=USER, queue=WORKER_TYPE,
                               **get_in_param_dict(t, year))
            job_id, status = r.id, r.status
        except Exception as e:
            job_id, status = None, f'error: {e}'
        rows.append({'dps_num': i, 'tile_num': t, 'submit_time': datetime.datetime.now().strftime('%Y-%m-%d-%H-%M-%S'),
                     'algo_id': ALGO_ID, 'user': USER, 'worker_type': WORKER_TYPE, 'job_id': job_id,
                     'submit_status': status, 'IDENTIFIER': identifier})
        if i in (1, len(tiles)) or i % 500 == 0:
            print(f'  {i}/{len(tiles)} tile {t}: {status}')
    df = pd.DataFrame(rows)
    now = pd.Timestamp.now().strftime('%Y%m%d%H%M')
    fn = f"{DPS_SUBMISSION_RESULTS_DIR}/DPS_{ALGO_ID}_{MAAP_VERSION}_{identifier.replace('/', '_')}_submission_results_{len(df)}_{now}.csv"
    df.to_csv(fn, index=False)
    print(f'  saved {fn}')
    return df

In [ ]:
TINDEX_TYPE = 'ATL08_filt'      # build_tindex type for parquet outputs; also written as ATL08_filt_extract_tindex_master.csv

def tindex_paths(identifier):
    d = f'{TINDEX_ROOT}/{identifier}'
    return {'dir': d, 'csv': f'{d}/{TINDEX_TYPE}_tindex_master.csv', 'extract_csv': f'{d}/ATL08_filt_extract_tindex_master.csv'}

def read_tindex_tiles(identifier):
    fn = tindex_paths(identifier)['csv']
    return set(pd.read_csv(fn).tile_num.astype(int)) if os.path.exists(fn) else set()

def dps_run_dates(identifier):
    """Years, months and day range of every DPS run-date folder: dps_output/<alg>/<version>/<identifier>/YYYY/MM/DD."""
    root = f'{DPS_OUTPUT_ROOT}/{identifier}'
    years, months, days = set(), set(), set()
    if os.path.isdir(root):
        for y in os.listdir(root):
            if not (y.isdigit() and len(y) == 4 and os.path.isdir(f'{root}/{y}')):
                continue
            for m in os.listdir(f'{root}/{y}'):
                if not (m.isdigit() and os.path.isdir(f'{root}/{y}/{m}')):
                    continue
                for d in os.listdir(f'{root}/{y}/{m}'):
                    if d.isdigit():
                        years.add(y); months.add(m); days.add(int(d))
    return sorted(years), sorted(months), (min(days), max(days)) if days else (1, 31)

def rebuild_tindex(identifier, ENV='pangeo', DEBUG=False,
                   SCRIPT=f'{ICESAT2_BOREAL_REPO_PATH}/lib/build_tindex_master_refactored.py'):
    """Back up, rebuild from ALL run dates (the script overwrites the tindex with only what it finds), verify the tile count."""
    tp = tindex_paths(identifier)
    years, months, (d_min, d_max) = dps_run_dates(identifier)
    if not years:
        print(f'  no DPS output folders under {DPS_OUTPUT_ROOT}/{identifier}; skipping')
        return None
    n_before = len(read_tindex_tiles(identifier))
    stamp = pd.Timestamp.now().strftime('%Y%m%d%H%M%S')
    bdir = f"{tp['dir']}/backup_{stamp}"
    for fn in (tp['csv'], tp['extract_csv']):
        if os.path.exists(fn):
            os.makedirs(bdir, exist_ok=True)
            shutil.copy2(fn, bdir)
    os.makedirs(tp['dir'], exist_ok=True)
    cmd = [f'/opt/conda/envs/{ENV}/bin/python', SCRIPT,
           '-t', TINDEX_TYPE, '-alg_name', ALGO_ID, '--user', USER,
           '--dps_identifier', f'{MAAP_VERSION}/{identifier}',
           '-y_list', *years, '-m_list', *months, '-d_min', str(d_min), '-d_max', str(d_max),
           '--outdir', tp['dir'], '-boreal_tile_index_path', boreal_tile_index_path, '--RETURN_DUPS']
    if DEBUG:
        cmd.append('--DEBUG')
    print(f'  run dates: years {years} months {months} days {d_min}-{d_max}')
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode != 0:
        print(res.stdout[-3000:], res.stderr[-3000:])
        raise RuntimeError(f'{os.path.basename(SCRIPT)} failed for {identifier}; backup in {bdir}')
    if DEBUG:
        print(res.stdout[-3000:])
    shutil.copy2(tp['csv'], tp['extract_csv'])     # the name mosaiclib.ATL08_FILT_EXTRACT_TINDEX_FN_DICT uses
    n_after = len(read_tindex_tiles(identifier))
    flag = 'OK' if n_after >= n_before else '!! SHRANK — restore with restore_tindex(identifier, backup_dir)'
    print(f'  tiles before {n_before} -> after {n_after}  {flag}   (backup: {bdir if n_before else "none"})')
    return {'identifier': identifier, 'n_before': n_before, 'n_after': n_after, 'backup_dir': bdir,
            'mosaiclib_entry': tp['extract_csv'].replace('/projects/my-public-bucket', f's3://maap-ops-workspace/shared/{USER}')}

def restore_tindex(identifier, backup_dir):
    for fn in glob.glob(f'{backup_dir}/*'):
        shutil.copy2(fn, tindex_paths(identifier)['dir'])
    print('restored', identifier, 'from', backup_dir)

## 2. Band check: what each covariate's columns will be called
One sample tile per year (the first tile with all covariates). Check that:
* every `sar` band has a `column` from the expected list (`how` = `S1 canonical` means it was renamed automatically),
* nothing is listed under **problems**,
* S1 `units` match the reference stacks the models were trained on (power, values ~0.01–0.5). A **dB** stack (negative values) needs converting before it can be used with the existing models.

In [ ]:
SAMPLE_TILE = None    # None = first tile per year that has every covariate; or set a tile_num
band_tables, problems = {}, {}
for y in YEAR_LIST:
    inputs = list_inputs(y)
    covs = {c: tindex_tiles(fn) for c, fn in covar_tindexes(y).items() if fn}
    cand = sorted(set(inputs).intersection(*covs.values())) if covs else []
    t = SAMPLE_TILE if SAMPLE_TILE is not None else (cand[0] if cand else (sorted(inputs)[0] if inputs else None))
    if t is None:
        problems[y] = {'atl08': f'no input parquets under {ATL08_IN_ROOT}/{y}'}; continue
    band_tables[y], problems[y] = band_check(y, t)
    band_tables[y].insert(0, 'tile_num', t)
    print(f'{y}: {len(inputs)} ATL08 inputs | sample tile {t} | problems: {problems[y] or "none"}')

pd.set_option('display.max_rows', 200)
pd.concat([d.assign(year=y) for y, d in band_tables.items() if len(d)], ignore_index=True).query("covar == 'sar'") if band_tables else None

In [ ]:
# S1 reference (the stacks the models were trained with): names and value levels on one tile
ref_tile = sorted(tindex_tiles(SAR_REFERENCE_TINDEX))[0]
ref_stack = stack_for_tile(SAR_REFERENCE_TINDEX, ref_tile)
ref = extract_atl08_covars.resolve_bandnames(ref_stack)[1]
ref['median_value'] = band_values_sample(ref_stack)
print('reference:', ref_stack)
ref

## 3. Test one tile (optional, runs here, not on DPS)
Same code DPS will run. Shows the new columns and the share of ATL08 rows that got a value from each covariate: S1 often has lower coverage than HLS.

In [ ]:
RUN_TEST = False
TEST_YEAR, TEST_TILE = YEAR_LIST[0], None      # None = the band-check sample tile
if RUN_TEST:
    t = TEST_TILE or int(band_tables[TEST_YEAR].tile_num.iloc[0])
    p = get_in_param_dict(t, TEST_YEAR)
    os.makedirs(TEST_OUTDIR, exist_ok=True)
    df = extract_atl08_covars.extract_atl08_covars(p['s3_atl08_gdf_fn'], p['tindex_fn_list'].split(), TEST_OUTDIR, RETURN_DF=True,
                                                   rename_map=extract_atl08_covars.parse_rename_map(p['band_rename']))
    new_cols = [c for c in band_tables[TEST_YEAR].column if c in df.columns]
    display(pd.DataFrame({'pct_with_value': (100 * df[new_cols].notna().mean()).round(1),
                          'median': df[new_cols].median().round(4)}))

## 4. Status: which tiles each year has and is missing
Outputs are read from the DPS output folders directly, so this is current even before the tindex is rebuilt.

In [ ]:
INPUTS = {y: set(list_inputs(y)) for y in YEAR_LIST}
TILES  = {y: sorted(INPUTS[y] & set(map(int, TILE_SET))) if TILE_SET is not None else sorted(INPUTS[y]) for y in YEAR_LIST}
HAVE   = {y: set(list_outputs(i)) for y, i in IDENTIFIERS.items()}
status = pd.DataFrame([{'year': y, 'identifier': IDENTIFIERS[y], 'n_atl08_inputs': len(INPUTS[y]), 'n_target': len(TILES[y]),
                        'n_have': len(set(TILES[y]) & HAVE[y]), 'n_missing': len(set(TILES[y]) - HAVE[y]),
                        'n_in_tindex': len(read_tindex_tiles(IDENTIFIERS[y]))} for y in YEAR_LIST])
status

In [ ]:
# Optional: map of missing tiles for one year
SHOW_MAP_YEAR = YEAR_LIST[0]
boreal_tiles = gpd.read_file(boreal_tile_index_path)
boreal_tiles[boreal_tiles.tile_num.isin(set(TILES[SHOW_MAP_YEAR]) - HAVE[SHOW_MAP_YEAR])].explore(color='red', tooltip='tile_num')

## 5. Fails: latest job per tile from the saved submission logs
Only tiles whose **most recent** job for the identifier failed (or never got a job id), and that still have no output. Cached to JSON so `DO_FAILS` works when you come back without re-querying (`REFRESH_FAILS = False`).

In [ ]:
FAILS_JSON = f"{PLAN_DIR}/fails_{ALGO_ID}_{MAAP_VERSION}_atl08_{ATL08_VERSION}.json"
if DO_FAILS and not REFRESH_FAILS and os.path.exists(FAILS_JSON):
    FAILED = {k: set(v) for k, v in json.load(open(FAILS_JSON)).items()}
    print('loaded cached fails from', FAILS_JSON)
elif DO_FAILS:
    FAILED, job_tables = {}, {}
    for y, ident in IDENTIFIERS.items():
        jobs = latest_jobs(ident)
        if jobs.empty:
            FAILED[y] = set(); print(f'{ident}: no submission logs'); continue
        jt = job_status_table(jobs)
        job_tables[y] = jt
        bad = jt.status.isin(['Failed', 'SubmitFailed', 'Offline', 'Deleted'])
        FAILED[y] = set(jt.tile_num[bad].astype(int)) - HAVE[y]
        print(f'{ident}: latest jobs {len(jt)} | ' + ', '.join(f'{k} {v}' for k, v in jt.status.value_counts().items())
              + f' | failed & no output: {len(FAILED[y])}')
    json.dump({y: sorted(v) for y, v in FAILED.items()}, open(FAILS_JSON, 'w'))
else:
    FAILED = {y: set() for y in YEAR_LIST}
    print('DO_FAILS = False: fails not computed')

In [ ]:
# Why did a job fail? (first few failed jobs of one year)
SHOW_FAIL_YEAR = YEAR_LIST[0]
if DO_FAILS and SHOW_FAIL_YEAR in job_tables:
    for job_id in job_tables[SHOW_FAIL_YEAR].query("status == 'Failed'").job_id.head(3):
        print(job_id, '\n\t', str(maap.getJobResult(job_id))[-1500:])

## 6. Plan: what will be submitted
Tiles without a covariate for that year (e.g. no HLS composite, or no S1 when `SAR_REQUIRED = True`) are listed as not runnable and are **not** submitted, so they don't fail repeatedly. Review this table before setting `DRY_RUN = False`.

In [ ]:
if not (DO_MISSING or DO_FAILS) and not CONFIRM_FULL_RUN:
    raise ValueError('Both DO_MISSING and DO_FAILS are False (full re-run). Set CONFIRM_FULL_RUN = True if intended.')
for y in YEAR_LIST:
    if problems.get(y):
        print(f'!! {y}: band check problems {problems[y]} — fix before submitting (section 2)')

plan = pd.concat([build_plan(y, IDENTIFIERS[y], TILES[y], HAVE[y], FAILED.get(y, set())) for y in YEAR_LIST], ignore_index=True)
plan = plan.astype({'tile_num': int, 'is_missing': bool, 'is_failed': bool, 'has_output': bool, 'runnable': bool})
plan_summary = (plan.groupby(['year', 'identifier'])
                    .agg(n_selected=('tile_num', 'size'), n_missing=('is_missing', 'sum'), n_failed=('is_failed', 'sum'),
                         n_not_runnable=('runnable', lambda s: int((~s).sum())), n_to_submit=('runnable', 'sum'),
                         n_without_sar=('n_covars', lambda s: int((s == 2).sum())))
                    .reset_index())
PLAN_FN = f"{PLAN_DIR}/plan_{ALGO_ID}_{MAAP_VERSION}_atl08_{ATL08_VERSION}_{MODE}_{pd.Timestamp.now():%Y%m%d%H%M}.csv"
plan.to_csv(PLAN_FN, index=False)
print('plan saved:', PLAN_FN, '| total to submit:', int(plan.runnable.sum()))
plan_summary

In [ ]:
# Not-runnable tiles and why (fix inputs first, or accept as permanent gaps)
plan[~plan.runnable].groupby(['year', 'reason_not_runnable']).tile_num.agg(['size', lambda s: sorted(s)[:20]]).reset_index()

## 7. Submit (only if `DRY_RUN = False`)

In [ ]:
SUBMITTED = {}
n_total = int(plan.runnable.sum())
bad_years = [y for y in YEAR_LIST if problems.get(y) and (plan.year == y).any() and plan[plan.year == y].runnable.any()]
if DRY_RUN:
    print(f'DRY_RUN: would submit {n_total} jobs. Set DRY_RUN = False in section 1 and re-run from the top.')
elif bad_years:
    raise ValueError(f'Band check problems for {bad_years} (section 2). Fix them, or drop those years from YEAR_LIST.')
elif n_total > MAX_JOBS:
    raise ValueError(f'{n_total} jobs > MAX_JOBS={MAX_JOBS}. Raise MAX_JOBS if intended.')
else:
    for y, ident in IDENTIFIERS.items():
        tiles = plan[(plan.year == y) & plan.runnable].tile_num.tolist()
        print(f'\n{ident}: {len(tiles)} jobs')
        if tiles:
            SUBMITTED[y] = submit_identifier(ident, y, tiles)

## 8. Assess the submission just made
Re-run this cell until nothing is `Accepted`/`Running`. To assess an older run in a new session, set `DO_FAILS = True` and run section 5, which reads the saved logs.

In [ ]:
if SUBMITTED:
    assess = pd.concat([job_status_table(df) for df in SUBMITTED.values()], ignore_index=True)
    display(assess.groupby(['IDENTIFIER', 'status']).size().unstack(fill_value=0))
else:
    print('Nothing submitted in this session.')

## 9. Rebuild the tindex (after jobs finish)
Uses **all** run-date folders found for each identifier, backs up the existing tindex first, and reports before/after tile counts. Add the printed `mosaiclib_entry` paths to `ATL08_FILT_EXTRACT_TINDEX_FN_DICT` (e.g. key `'2020_v007'`) so `DPS_run_boreal_v2` can use them.

In [ ]:
DO_REBUILD_TINDEX = False
if DO_REBUILD_TINDEX:
    rebuilt = [rebuild_tindex(i) for i in IDENTIFIERS.values()]
    display(pd.DataFrame([r for r in rebuilt if r]))

In [ ]:
# Check one output: same columns as the band check, and how many rows got S1 values
CHECK_YEAR = YEAR_LIST[0]
outs = list_outputs(IDENTIFIERS[CHECK_YEAR])
if outs:
    t, fn = next(iter(sorted(outs.items())))
    o = gpd.read_parquet(fn)
    cov_cols = [c for c in sum(EXPECTED_BANDS.values(), []) if c in o.columns]
    print(fn, o.shape)
    print('expected columns missing:', [c for c in sum(EXPECTED_BANDS.values(), []) if c not in o.columns and not c.startswith('n_')])
    display((100 * o[cov_cols].notna().mean()).round(1).rename('pct_with_value').to_frame().T)

---
## Appendix: reference calls (not part of the run)

In [ ]:
SHOW_REFERENCE = False
if SHOW_REFERENCE:
    print([a for a in maap.listAlgorithms().json()['algorithms'] if ALGO_ID in a['type']])
    print(maap.getQueues().json()['queues'])
    # Register the algorithm (once per MAAP_VERSION; push the git tag of the same name first):
    # maap.register_algorithm_from_yaml_file(f'{ICESAT2_BOREAL_REPO_PATH}/dps/registered/run_extract_atl08_covars.yml').text
    # Inspect the full parameter dict for one tile:
    y = YEAR_LIST[0]
    print(json.dumps(get_in_param_dict(sorted(list_inputs(y))[0], y), indent=1))